_Célula 1_

# Gráficos combinados: Energia Solar (UFV) + Eólica
Junta `UFV_consolidado.csv` e `eolica_consolidado.csv` (mesmo formato: `year`, `area_ha`, `tipo_region`, `nome_region`, `sigla_region`) em um único `df`, marcando a origem de cada linha em `tipo_energia` (`UFV` ou `Eólica`). Repete as mesmas análises por país, bioma e estado, agora com as áreas das duas fontes de energia somadas, e adiciona uma seção final comparando a participação de cada tipo de energia no total.

In [1]:
# Célula 2
import pandas as pd
import matplotlib.pyplot as plt
import kaleido
import numpy as np
import plotly.graph_objects as go
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
import textwrap

from_drive = False
salvar_grafico = True

if from_drive:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    CAMINHO_UFV = '/content/drive/MyDrive/DL_fotovoltaica/UFV_consolidado.csv'
    CAMINHO_EOLICA = '/content/drive/MyDrive/DL_fotovoltaica/eolica_consolidado.csv'
else:
    CAMINHO_UFV = 'UFV_consolidado.csv'
    CAMINHO_EOLICA = 'eolica_consolidado.csv'

df_ufv = pd.read_csv(CAMINHO_UFV)
df_ufv['tipo_energia'] = 'UFV'

df_eolica = pd.read_csv(CAMINHO_EOLICA)
df_eolica['tipo_energia'] = 'Eólica'

# Junta as duas fontes num único DataFrame — o restante do notebook agrega
# por tipo_region (bioma/estado/pais) somando as duas energias, exceto na
# seção final "Por tipo", que olha a quebra UFV x Eólica.
df = pd.concat([df_ufv, df_eolica], ignore_index=True)
df.head()

,year,version,area_ha,tipo_region,nome_region,sigla_region,tipo_energia
0,2018,0-4-12-spt-4,0.617504,bioma,Amazônia,AMZ,UFV
1,2019,0-4-12-spt-4,16.561835,bioma,Amazônia,AMZ,UFV
2,2020,0-4-12-spt-4,20.086309,bioma,Amazônia,AMZ,UFV
3,2021,0-4-12-spt-4,24.763521,bioma,Amazônia,AMZ,UFV
4,2022,0-4-12-spt-4,25.998181,bioma,Amazônia,AMZ,UFV


In [2]:
# Célula 3
# Paleta sequencial usada nos gráficos (clara = valor menor/ano mais antigo,
# escura = valor maior/ano mais recente).
CMAP_CINZA = mcolors.LinearSegmentedColormap.from_list(
    'combinado_cinza', ['#a8a5a5', '#948f8f', '#807b7b', '#6c6767', '#585353', '#443f40', '#302b2c']
)


def rampa_cores(n: int, inverso: bool = False):
    """`n` cores igualmente espaçadas na paleta sequencial (clara → escura)."""
    posicoes = np.linspace(0.05, 0.95, n)
    if inverso:
        posicoes = posicoes[::-1]
    return [CMAP_CINZA(p) for p in posicoes]


def formata_ptbr(valor, casas: int = 0) -> str:
    """Formata número no padrão brasileiro (ponto como separador de milhar)."""
    s = f'{valor:,.{casas}f}'
    return s.replace(',', '_').replace('.', ',').replace('_', '.')


def quebra_nome(nome: str, largura: int = 12) -> str:
    """Quebra nomes de região longos em 2 linhas para caber no eixo X."""
    return '\n'.join(textwrap.wrap(nome, width=largura, break_long_words=False))


FORMATADOR_PTBR = FuncFormatter(lambda v, _: formata_ptbr(v))

print('Utilitários de estilo definidos.')

Utilitários de estilo definidos.


In [3]:
# Célula 4
# Requer o pacote `kaleido` instalado (pip install -U kaleido) para exportar
# as figuras Plotly como imagem.
try:
    from google.colab import files as colab_files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False


def salvar_grafico_alta_resolucao(fig, nome_arquivo, dpi=300, formato='png', baixar=True):
    """Salva um gráfico Plotly (`fig`) em alta resolução (300 ou 450 dpi) e,
    se `baixar=True` e estiver rodando no Colab, dispara o download do
    arquivo.

    O Plotly/kaleido não trabalha com DPI diretamente — a resolução da
    imagem exportada é controlada por `scale` (multiplicador do
    width/height definidos no layout da figura). Aqui `scale` é calculado
    como dpi/96, usando 96 dpi como referência padrão de tela.
    """
    escala = dpi / 96
    caminho = nome_arquivo if nome_arquivo.lower().endswith(f'.{formato}') else f"{nome_arquivo}.{formato}"
    fig.write_image(caminho, scale=escala)
    print(f"Gráfico salvo em '{caminho}' (dpi≈{dpi}, scale={escala:.2f})")

    if baixar and EM_COLAB:
        colab_files.download(caminho)


# uso: salvar_grafico_alta_resolucao(fig, 'evolucao_area_combinada_por_bioma', dpi=300)
# ou, para mais nitidez: dpi=450

_Célula 5_

## Resumo
Evolução da área por bioma somando UFV + Eólica.

In [4]:
# Célula 6
df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
totais_anuais = pivot_bioma.sum(axis=1)
ultimo_ano_bioma = pivot_bioma.index.max()

cores_map = {'Caatinga': '#4a4747', 'Cerrado': '#666363', 'Mata Atlântica': '#807b7b', 'Amazônia': '#9a9595', 'Pampa': '#b4b0b0'}

# Os 2 biomas com maior área no último ano ganham rótulo (valor + %) em todas
# as barras, para acompanhar o histórico; os demais ficam sem rótulo — quais
# são muda automaticamente conforme os dados (não fixo em nomes).
BIOMAS_HISTORICO = pivot_bioma.loc[ultimo_ano_bioma].sort_values(ascending=False).index[:2].tolist()

fig = go.Figure()

for bioma in pivot_bioma.columns:
    textos = []
    for ano, v in zip(pivot_bioma.index, pivot_bioma[bioma]):
        if bioma in BIOMAS_HISTORICO and v > 0:
            valor_k = f"{v/1000:.1f} K".replace('.', ',')
            pct = (v / totais_anuais.loc[ano]) * 100
            textos.append(f"{valor_k}<br>{pct:.0f}%")
        else:
            textos.append("")

    fig.add_trace(go.Bar(
        x=pivot_bioma.index, y=pivot_bioma[bioma], name=bioma,
        marker_color=cores_map.get(bioma, '#ccc'),
        text=textos, textposition='inside', insidetextanchor='middle',
        textfont=dict(color='white', size=14),
        hovertemplate='<b>Bioma:</b> ' + bioma + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

# Rótulo do total (UFV + Eólica somados) acima de cada barra, em todos os anos.
textos_totais = [f"<b>{v/1000:.1f} K</b>".replace('.', ',') for v in totais_anuais]
fig.add_trace(go.Scatter(
    x=totais_anuais.index, y=totais_anuais * 1.05, mode='text', text=textos_totais,
    textposition='top center', textfont=dict(size=15, color='black'),
    showlegend=False, hoverinfo='skip',
    cliponaxis=False,
))

fig.update_layout(barmode='stack', title_text="Evolução da Área Combinada (UFV + Eólica) por Bioma",
    template="plotly_white", width=900, height=650,
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'),
    margin=dict(t=80, b=150, l=60, r=60), xaxis=dict(tickmode='linear'), yaxis=dict(range=[0, totais_anuais.max() * 1.3]))
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_area_combinada_por_bioma', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_area_combinada_por_bioma.png' (dpi≈450, scale=4.69)


In [5]:
# Célula 6A
# Variante em gráfico de área empilhada (em vez de barras) do mesmo painel de
# biomas, para os dados combinados (UFV + Eólica) — 5 biomas (a Amazônia só
# existe em UFV).
df_bioma_area = df[df['tipo_region'] == 'bioma']
pivot_bioma_area = df_bioma_area.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
totais_anuais_area = pivot_bioma_area.sum(axis=1)

cores_map_area = {'Caatinga': '#4a4747', 'Cerrado': '#666363', 'Mata Atlântica': '#807b7b', 'Amazônia': '#9a9595', 'Pampa': '#b4b0b0'}

fig_area = go.Figure()

for bioma in pivot_bioma_area.columns:
    fig_area.add_trace(go.Scatter(
        x=pivot_bioma_area.index, y=pivot_bioma_area[bioma], name=bioma,
        mode='lines', stackgroup='biomas',
        line=dict(width=0.5, color=cores_map_area.get(bioma, '#ccc')),
        fillcolor=cores_map_area.get(bioma, '#ccc'),
        hovertemplate='<b>Bioma:</b> ' + bioma + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

# Bloco de texto com o total do ano (UFV + Eólica somados), ancorado acima da
# área empilhada.
y_texto_area = totais_anuais_area * 1.08
textos_totais_area = [f"<b>{v/1000:.1f} K</b>".replace('.', ',') for v in totais_anuais_area]

fig_area.add_trace(go.Scatter(
    x=totais_anuais_area.index, y=y_texto_area, mode='text', text=textos_totais_area,
    textposition='top center', textfont=dict(size=14, color='black'),
    showlegend=False, hoverinfo='skip',
    cliponaxis=False,
))

fig_area.update_layout(
    title_text="Evolução da Área Combinada (UFV + Eólica) por Bioma (Área)", template="plotly_white",
    width=700, height=680,
    # Legenda em 2 colunas: orientação horizontal + cada item ocupando metade
    # da largura disponível, então só cabem 2 por linha antes de quebrar
    # (com 5 biomas, isso dá 3 linhas de até 2 itens).
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=14), entrywidth=0.5, entrywidthmode='fraction'),
    margin=dict(t=100, b=180, l=60, r=150), xaxis=dict(tickmode='linear'),
    yaxis=dict(range=[0, totais_anuais_area.max() * 1.3])
)
fig_area.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig_area, 'evolucao_area_combinada_por_bioma_area', dpi=450)

Salvando o grafico com DPI 450


Gráfico salvo em 'evolucao_area_combinada_por_bioma_area.png' (dpi≈450, scale=4.69)


In [6]:
# Célula 6A
# Variante em gráfico de área empilhada (em vez de barras) do mesmo painel de biomas.
df_bioma_area = df[df['tipo_region'] == 'bioma']
pivot_bioma_area = df_bioma_area.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
totais_anuais_area = pivot_bioma_area.sum(axis=1)

cores_map_area = {'Caatinga': "#1b1a1a", 'Cerrado': "#3C3C3C", 'Mata Atlântica': "#7A7878", 'Pampa': "#C1C0C0"}
BIOMAS_MINORITARIOS_AREA = ['Cerrado', 'Mata Atlântica', 'Pampa']

fig_area = go.Figure()

for bioma in pivot_bioma_area.columns:
    # Rótulo da Caatinga (valor em K nos extremos 2016/2025) fica oculto —
    # só o bloco de total por ano (abaixo) continua visível.
    textos = ["" for _ in pivot_bioma_area.index]

    fig_area.add_trace(go.Scatter(
        x=pivot_bioma_area.index, y=pivot_bioma_area[bioma], name=bioma,
        mode='lines+text', stackgroup='biomas',
        line=dict(width=0.5, color=cores_map_area.get(bioma, '#ccc')),
        fillcolor=cores_map_area.get(bioma, '#ccc'),
        text=textos, textposition='bottom center',
        textfont=dict(color='white', size=16),
        hovertemplate='<b>Bioma:</b> ' + bioma + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

# Bloco de texto com o total do ano, ancorado acima da área empilhada — apenas
# o total em negrito para todos os anos (rótulos de Cerrado, Mata Atlântica e
# Pampa ficam ocultos também em 2025).
y_texto_area = totais_anuais_area * 1.08

textos_totais_area = [f"<b>{total/1000:.1f} K</b>".replace('.', ',') for total in totais_anuais_area]

fig_area.add_trace(go.Scatter(
    x=totais_anuais_area.index, y=y_texto_area, mode='text', text=textos_totais_area,
    textposition='top center', textfont=dict(size=14, color='black'),
    showlegend=False, hoverinfo='skip',
    cliponaxis=False,
))

fig_area.update_layout(
    barmode='stack', title_text="Evolução da Área de Energia Eólica por Bioma (Área)", template="plotly_white",
    width=650, height=650,
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16), entrywidth=0.24, entrywidthmode='fraction'),
    margin=dict(t=100, b=150, l=60, r=150), xaxis=dict(tickmode='linear'),
    yaxis=dict(range=[0, totais_anuais_area.max() * 1.45])
)
fig_area.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig_area, 'evolucao_area_por_bioma_combinado_area', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_area_por_bioma_combinado_area.png' (dpi≈450, scale=4.69)


In [7]:
# Célula 7
ANO_INICIAL, ANO_FINAL = 2020, 2025


def calcula_crescimento(df, tipo_region, ano_inicial=ANO_INICIAL, ano_final=ANO_FINAL, coluna_grupo='nome_region'):
    """Área e % de crescimento entre `ano_inicial` e `ano_final` para cada
    valor de `coluna_grupo` (por padrão `nome_region`; também aceita
    `tipo_energia` para comparar UFV x Eólica) dentro do `tipo_region`
    informado ('bioma', 'estado' ou 'pais')."""
    sub = df[df['tipo_region'] == tipo_region]
    pivot = sub.pivot_table(index='year', columns=coluna_grupo, values='area_ha', aggfunc='sum').fillna(0)

    area_inicial = pivot.loc[ano_inicial]
    area_final = pivot.loc[ano_final]

    tabela = pd.DataFrame({
        coluna_grupo: area_inicial.index,
        f'area_{ano_inicial}': area_inicial.values,
        f'area_{ano_final}': area_final.values,
    })
    # np.where evita divisão por zero nas regiões que ainda não tinham área em ano_inicial
    tabela['crescimento_pct'] = np.where(
        tabela[f'area_{ano_inicial}'] > 0,
        (tabela[f'area_{ano_final}'] / tabela[f'area_{ano_inicial}'] - 1) * 100,
        np.nan,
    )
    return tabela.sort_values('crescimento_pct', ascending=False, na_position='last').reset_index(drop=True)


def formata_tabela_crescimento(tabela):
    tabela_fmt = tabela.copy()
    for col in tabela_fmt.columns:
        if col.startswith('area_'):
            tabela_fmt[col] = tabela_fmt[col].apply(formata_ptbr)
    tabela_fmt['crescimento_pct'] = tabela_fmt['crescimento_pct'].apply(
        lambda p: f"{p:.0f}%".replace('.', ',') if pd.notna(p) else "—"
    )
    return tabela_fmt


# Crescimento nacional combinado (país)
crescimento_pais = calcula_crescimento(df, 'pais')
pct_brasil = crescimento_pais['crescimento_pct'].iloc[0]
print(f"Crescimento da área combinada (UFV + Eólica) no Brasil entre {ANO_INICIAL} e {ANO_FINAL}: "
      + f"{pct_brasil:.0f}%".replace('.', ','))
display(formata_tabela_crescimento(crescimento_pais))

# Crescimento por estado (combinado)
print(f"\nCrescimento por estado entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'estado')))

# Crescimento por bioma (combinado)
print(f"\nCrescimento por bioma entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'bioma')))

Crescimento da área combinada (UFV + Eólica) no Brasil entre 2020 e 2025: 190%


,nome_region,area_2020,area_2025,crescimento_pct
0,Brasil,24.791,71.912,190%



Crescimento por estado entre 2020 e 2025:


,nome_region,area_2020,area_2025,crescimento_pct
0,Minas Gerais,1.380,16.258,1078%
1,Pernambuco,912,4.364,378%
2,Paraíba,746,2.595,248%
3,Tocantins,12,43,245%
4,São Paulo,643,2.077,223%
5,Ceará,2.674,7.080,165%
6,Rio Grande do Norte,5.567,13.544,143%
7,Bahia,7.092,15.274,115%
8,Piauí,4.212,8.390,99%
9,Rio Grande do Sul,1.102,1.703,54%



Crescimento por bioma entre 2020 e 2025:


,nome_region,area_2020,area_2025,crescimento_pct
0,Cerrado,5.010,16.065,221%
1,Caatinga,17.770,51.604,190%
2,Mata Atlântica,899,2.517,180%
3,Pampa,1.094,1.698,55%
4,Amazônia,20,31,53%


In [6]:
# Célula 8
df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano = int(pivot_bioma.index.max())
serie_ultimo_ano = pivot_bioma.loc[ultimo_ano].sort_values(ascending=False)

cores_map = {'Caatinga': '#4a4747', 'Cerrado': '#666363', 'Mata Atlântica': '#807b7b', 'Amazônia': '#9a9595', 'Pampa': '#b4b0b0'}

# rotation pode precisar de ajuste visual (como nos outros dois notebooks) se
# as fatias pequenas colidirem com o título ou entre si.
fig = go.Figure(data=[go.Pie(
    labels=serie_ultimo_ano.index,
    values=serie_ultimo_ano.values,
    marker=dict(colors=[cores_map.get(b) for b in serie_ultimo_ano.index]),
    textinfo='percent+label+value',
    rotation=90,
    texttemplate='%{label}<br>%{percent:.0%}<br>%{value:,.2f} ha',
    insidetextfont=dict(size=16),
    outsidetextfont=dict(size=16),
    automargin=True,
    hovertemplate='<b>Bioma:</b> %{label}<br><b>Área:</b> %{value:,.2f} ha<extra></extra>'
)])
fig.update_layout(
    title_text=f"Participação por Bioma na Área Combinada (UFV + Eólica) — {ultimo_ano}",
    title_font=dict(size=22), template="plotly_white", width=800, height=700,
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5, entrywidth=0.3, entrywidthmode='fraction', font=dict(size=16)),
    margin=dict(t=80, b=100, l=50, r=50)
)
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'proporcao_area_combinada_por_bioma', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'proporcao_area_combinada_por_bioma.png' (dpi≈450, scale=4.69)


In [9]:
# Célula 8A
# Anéis concêntricos (não fatias de uma mesma pizza): cada bioma ocupa uma
# faixa de raio própria, do maior para o menor (mais externo -> mais
# interno), e todos os arcos partem do mesmo ângulo inicial (0°/12h), com o
# comprimento proporcional à participação do bioma no total combinado
# (UFV + Eólica). Por isso os arcos ficam com tamanhos bem diferentes entre
# si (esperado).
import plotly.graph_objects as go

df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano = int(pivot_bioma.index.max())
serie_ultimo_ano = pivot_bioma.loc[ultimo_ano].sort_values(ascending=False)
total_ano = serie_ultimo_ano.sum()

cores_map = {'Caatinga': '#4a4747', 'Cerrado': '#666363', 'Mata Atlântica': '#807b7b', 'Amazônia': '#9a9595', 'Pampa': '#b4b0b0'}

# Ordem dos anéis (do mais externo ao mais interno) calculada dinamicamente a
# partir dos dados — igual é feito para BIOMAS_HISTORICO na Célula 6 — em vez
# de fixar os 4 biomas da Eólica, aqui entram os 5 biomas do dado combinado
# (a Amazônia só existe em UFV).
ORDEM_ANEIS = serie_ultimo_ano.index.tolist()
N = len(ORDEM_ANEIS)
ESPACO = 0.02  # respiro entre um anel e o próximo
ESPESSURA = (1 - (N - 1) * ESPACO) / N  # espessura de cada anel, para o conjunto ocupar de r=0 até r=1

fig_aneis = go.Figure()
for i, bioma in enumerate(ORDEM_ANEIS):
    valor = serie_ultimo_ano.get(bioma, 0)
    frac = valor / total_ano
    arco_graus = frac * 360
    base_raio = 1 - i * (ESPESSURA + ESPACO) - ESPESSURA

    fig_aneis.add_trace(go.Barpolar(
        r=[ESPESSURA],
        theta=[arco_graus / 2],  # todos os arcos começam em 0°; o centro do arco é a metade do seu próprio comprimento
        width=[arco_graus],
        base=[base_raio],
        name=bioma,
        marker=dict(color=cores_map.get(bioma, '#ccc'), line=dict(color='white', width=1)),
        hovertemplate=f'<b>Bioma:</b> {bioma}<br><b>Área:</b> {valor:,.2f} ha<br><b>Participação:</b> {frac*100:.1f}%<extra></extra>',
    ))

# --- rótulos: lista empilhada à esquerda do anel, texto grande e em negrito,
# tamanho decrescente conforme a posição no ranking (não fixo por nome de
# bioma, já que o combinado tem 5 biomas em vez dos 4 da Eólica) ---
TAMANHOS_FONTE = np.linspace(30, 15, N)

anotacoes = []
Y_INICIAL, PASSO_Y = 0.90, 0.72 / max(N - 1, 1)
for i, bioma in enumerate(ORDEM_ANEIS):
    valor = serie_ultimo_ano.get(bioma, 0)
    frac = valor / total_ano
    anotacoes.append(dict(
        x=0.04, y=Y_INICIAL - i * PASSO_Y, xref='paper', yref='paper',
        xanchor='left', align='left',
        text=f"<b>{bioma} {frac*100:.0f}%</b>",
        showarrow=False,
        font=dict(size=int(TAMANHOS_FONTE[i]), color='#1b1a1a'),
    ))

fig_aneis.update_layout(
    title_text=f"Participação por Bioma na Área Combinada (UFV + Eólica) — {ultimo_ano}",
    title_font=dict(size=24),
    template="plotly_white",
    width=900, height=800,
    showlegend=False,
    annotations=anotacoes,
    polar=dict(
        # anel deslocado para a direita, deixando espaço à esquerda para os rótulos
        domain=dict(x=[0.38, 0.98], y=[0.05, 0.95]),
        radialaxis=dict(visible=False, range=[0, 1]),
        angularaxis=dict(visible=False, rotation=90, direction='clockwise'),
    ),
    margin=dict(t=90, b=30, l=30, r=30),
)

fig_aneis.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig_aneis, 'proporcao_area_por_bioma_combinado_aneis_concentricos', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'proporcao_area_por_bioma_combinado_aneis_concentricos.png' (dpi≈450, scale=4.69)


_Célula 9_

## Estados
Evolução da área por estado somando UFV + Eólica — como o número de estados quase dobra em relação a cada gráfico isolado, mostra só os de maior área combinada.

In [7]:
# Célula 10
df_estado = df[df['tipo_region'] == 'estado']
pivot_estado = df_estado.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_estado = pivot_estado.index.max()
ordem_desc = pivot_estado.loc[ultimo_ano_estado].sort_values(ascending=False).index
pivot_estado = pivot_estado[ordem_desc]

# Mostra só os 10 estados com maior área combinada no ano mais recente — com
# as duas fontes de energia juntas o número de estados quase dobra e um
# gráfico com todos ficaria ilegível.
TOP_N_ESTADOS = 10
estados = pivot_estado.columns.tolist()[:TOP_N_ESTADOS]
anos = pivot_estado.index.tolist()

cores_anos_plotly = [f'rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})' for r, g, b, a in rampa_cores(len(anos))]
max_valor_estado = max(pivot_estado.loc[ano, e] for ano in anos for e in estados)

fig = go.Figure()

for i, ano in enumerate(anos):
    font_size = 14 if ano == ultimo_ano_estado else 1
    exibir_texto = ano == ultimo_ano_estado

    valores = [pivot_estado.loc[ano, e] for e in estados]
    textos = [formata_ptbr(v) if (v > 0 and exibir_texto) else "" for v in valores]

    fig.add_trace(
        go.Bar(
            x=estados,
            y=valores,
            name=str(ano),
            marker_color=cores_anos_plotly[i],
            text=textos,
            textposition='outside',
            textfont=dict(size=font_size, color='black'),
            textangle=-90,
            # Sem isso, o Plotly encolhe o texto das barras mais altas (perto
            # do topo do gráfico) para caber no espaço disponível.
            constraintext='none',
            hovertemplate='<b>Estado:</b> %{x}<br><b>Ano:</b> ' + str(ano) + '<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
        )
    )

fig.update_layout(
    title=f"Estados - Evolução da Área Combinada (UFV + Eólica) — Top {TOP_N_ESTADOS}, destaque {ultimo_ano_estado}",
    width=1050, height=750, template="plotly_white", barmode='group',
    legend=dict(
        orientation="h", y=-0.05, x=0.5, xanchor="center",
        font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'
    ),
    margin=dict(t=80, b=150, l=60, r=40),
    yaxis=dict(range=[0, max_valor_estado * 1.35])
)
fig.update_xaxes(tickangle=0)
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_anoXarea_combinada_por_estado', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_anoXarea_combinada_por_estado.png' (dpi≈450, scale=4.69)


_Célula 11_

### Participação percentual por estado (ano mais recente)
Percentual da área combinada (UFV + Eólica) de cada estado em relação ao total somado de todos os estados.

In [8]:
# Célula 12
df_estado_pct = df[df['tipo_region'] == 'estado']
pivot_estado_pct = df_estado_pct.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_pct = pivot_estado_pct.index.max()

serie_estado_pct = pivot_estado_pct.loc[ultimo_ano_pct].sort_values(ascending=False)
total_estados_pct = serie_estado_pct.sum()

tabela_pct_estado = pd.DataFrame({
    'estado': serie_estado_pct.index,
    'area_ha': serie_estado_pct.values,
})
tabela_pct_estado['percentual'] = tabela_pct_estado['area_ha'] / total_estados_pct * 100

# versão formatada (pt-BR) só para exibição — tabela_pct_estado continua numérica
tabela_pct_estado_fmt = tabela_pct_estado.copy()
tabela_pct_estado_fmt['area_ha'] = tabela_pct_estado_fmt['area_ha'].apply(formata_ptbr)
tabela_pct_estado_fmt['percentual'] = tabela_pct_estado_fmt['percentual'].apply(lambda p: f"{p:.1f}%".replace('.', ','))

display(tabela_pct_estado_fmt)

,estado,area_ha,percentual
0,Minas Gerais,16.258,"22,6%"
1,Bahia,15.274,"21,2%"
2,Rio Grande do Norte,13.544,"18,8%"
3,Piauí,8.390,"11,7%"
4,Ceará,7.080,"9,8%"
5,Pernambuco,4.364,"6,1%"
6,Paraíba,2.595,"3,6%"
7,São Paulo,2.077,"2,9%"
8,Rio Grande do Sul,1.703,"2,4%"
9,Maranhão,451,"0,6%"


_Célula 13_

### Estados agrupados — Top 4 (ano mais recente)
Soma da área combinada e percentual conjunto dos 4 estados com maior área (UFV + Eólica) em relação ao total geral — calculado dinamicamente, já que o ranking pode mudar em relação aos gráficos de cada fonte isolada.

In [ ]:
# Célula 14
df_estado_grupo = df[df['tipo_region'] == 'estado']
pivot_estado_grupo = df_estado_grupo.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_grupo = pivot_estado_grupo.index.max()

serie_estado_grupo = pivot_estado_grupo.loc[ultimo_ano_grupo].sort_values(ascending=False)
total_geral_estados = serie_estado_grupo.sum()

# Top 4 calculado dinamicamente (não fixo em nomes) — com UFV + Eólica juntos
# o ranking pode ser diferente do observado em cada gráfico isolado.
ESTADOS_GRUPO = serie_estado_grupo.index[:4].tolist()
area_grupo = serie_estado_grupo[ESTADOS_GRUPO].sum()
percentual_grupo = area_grupo / total_geral_estados * 100

tabela_grupo_estados = pd.DataFrame({
    'estados': [' + '.join(ESTADOS_GRUPO)],
    'area_ha': [area_grupo],
    'percentual': [percentual_grupo],
})

# versão formatada (pt-BR) só para exibição
tabela_grupo_estados_fmt = tabela_grupo_estados.copy()
tabela_grupo_estados_fmt['area_ha'] = tabela_grupo_estados_fmt['area_ha'].apply(formata_ptbr)
tabela_grupo_estados_fmt['percentual'] = tabela_grupo_estados_fmt['percentual'].apply(lambda p: f"{p:.1f}%".replace('.', ','))

display(tabela_grupo_estados_fmt)

_Célula 15_

## País (Brasil)
Área combinada (UFV + Eólica) no Brasil por ano.

In [ ]:
# Célula 16
df_pais = df[df['tipo_region'] == 'pais']
pivot_pais = df_pais.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
serie_brasil = pivot_pais['Brasil']

anos = serie_brasil.index.tolist()
valores = serie_brasil.values.tolist()

cores_br = [f'rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})' for r, g, b, a in rampa_cores(len(anos))]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=anos,
    y=valores,
    mode='lines+markers+text',
    line=dict(color='#666363', width=4),
    marker=dict(
        color=cores_br,
        size=12,
        line=dict(color='white', width=1.5)
    ),
    text=[formata_ptbr(v) for v in valores],
    textposition="top center",
    textfont=dict(size=11, color='#555'),
    hovertemplate='<b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
))

fig.update_layout(
    title="Área Combinada (UFV + Eólica) no Brasil por Ano",
    xaxis_title="Ano",
    yaxis_title="Área (ha)",
    template="plotly_white",
    width=800,
    height=550,
    xaxis=dict(
        tickmode='linear',
        range=[min(anos) - 0.5, max(anos) + 0.5]
    ),
    yaxis=dict(
        range=[0, max(valores) * 1.2],
        tickformat=".2s",
        hoverformat=",.2f"
    ),
    margin=dict(t=80, b=60, l=60, r=40)
)
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_area_combinada_pais', dpi=450)

_Célula 17_

## Por tipo (UFV x Eólica)
Última quebra pedida: em vez de somar as duas fontes, esta seção compara a contribuição de cada tipo de energia (solar/UFV x eólica) no total nacional.

In [ ]:
# Célula 18
df_pais_tipo = df[df['tipo_region'] == 'pais']
pivot_tipo = df_pais_tipo.pivot_table(index='year', columns='tipo_energia', values='area_ha', aggfunc='sum').fillna(0)
totais_tipo = pivot_tipo.sum(axis=1)

cores_tipo = {'UFV': '#4a4747', 'Eólica': '#b4b0b0'}

fig = go.Figure()

for tipo in pivot_tipo.columns:
    textos = []
    for ano, v in zip(pivot_tipo.index, pivot_tipo[tipo]):
        if v > 0:
            valor_k = f"{v/1000:.1f} K".replace('.', ',')
            pct = (v / totais_tipo.loc[ano]) * 100
            textos.append(f"{valor_k}<br>{pct:.0f}%")
        else:
            textos.append("")
    fig.add_trace(go.Bar(
        x=pivot_tipo.index, y=pivot_tipo[tipo], name=tipo,
        marker_color=cores_tipo.get(tipo, '#ccc'),
        text=textos, textposition='inside', insidetextanchor='middle',
        textfont=dict(color='white', size=14),
        hovertemplate='<b>Tipo:</b> ' + tipo + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

textos_totais = [f"<b>{v/1000:.1f} K</b>".replace('.', ',') for v in totais_tipo]
fig.add_trace(go.Scatter(
    x=totais_tipo.index, y=totais_tipo * 1.05, mode='text', text=textos_totais,
    textposition='top center', textfont=dict(size=15, color='black'),
    showlegend=False, hoverinfo='skip',
    cliponaxis=False,
))

fig.update_layout(barmode='stack', title_text="Evolução da Área no Brasil por Tipo de Energia (UFV x Eólica)",
    template="plotly_white", width=900, height=650,
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16)),
    margin=dict(t=80, b=120, l=60, r=60), xaxis=dict(tickmode='linear'), yaxis=dict(range=[0, totais_tipo.max() * 1.3]))
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_area_por_tipo_energia', dpi=450)

In [ ]:
# Célula 19
ultimo_ano_tipo = int(pivot_tipo.index.max())
serie_ultimo_ano_tipo = pivot_tipo.loc[ultimo_ano_tipo].sort_values(ascending=False)

fig = go.Figure(data=[go.Pie(
    labels=serie_ultimo_ano_tipo.index,
    values=serie_ultimo_ano_tipo.values,
    marker=dict(colors=[cores_tipo.get(t) for t in serie_ultimo_ano_tipo.index]),
    textinfo='percent+label+value',
    texttemplate='%{label}<br>%{percent:.0%}<br>%{value:,.2f} ha',
    insidetextfont=dict(size=18),
    outsidetextfont=dict(size=18),
    hovertemplate='<b>Tipo:</b> %{label}<br><b>Área:</b> %{value:,.2f} ha<extra></extra>'
)])
fig.update_layout(
    title_text=f"Participação por Tipo de Energia no Total Combinado ({ultimo_ano_tipo})",
    title_font=dict(size=22), template="plotly_white", width=700, height=600,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5, font=dict(size=16)),
    margin=dict(t=80, b=80, l=50, r=50)
)
fig.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'proporcao_area_por_tipo_energia', dpi=450)

In [ ]:
# Célula 20
print(f"Crescimento por tipo de energia entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'pais', coluna_grupo='tipo_energia')))

In [ ]:
# Célula 21